# Problem 49: No Match Detector

**Difficulty:** Medium | **Framework:** CrewAI

**Categories:** rag

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/no-match-detector).

## 1. Install Dependencies

In [ ]:
!pip install -q crewai crewai-tools

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must detect when retrieved documents are not relevant to the query
- A similarity score threshold or relevance check must be implemented
- The agent must respond with a clear 'I don't have information' message when no match is found
- The agent must not fabricate answers when retrieval quality is low


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from crewai import Agent, Task, Crew
from crewai.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings()

documents = [
    Document(page_content="Our return policy allows returns within 30 days of purchase."),
    Document(page_content="Shipping is free for orders over $50 within the continental US."),
    Document(page_content="Our customer support hours are 9 AM to 5 PM EST, Monday through Friday."),
]

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

# BUG: Always generates an answer regardless of retrieval quality
@tool
def search_knowledge_base(question: str) -> str:
    """Search the knowledge base for relevant information."""
    docs = retriever.invoke(question)
    return "\n".join([doc.page_content for doc in docs])

support_agent = Agent(
    role="Support Agent",
    goal="Answer customer questions using the knowledge base",
    backstory="You are a customer support agent. Answer questions using the search tool.",
    tools=[search_knowledge_base],
)

task = Task(
    description="What programming languages does your API support?",
    expected_output="Information about supported programming languages",
    agent=support_agent,
)

crew = Crew(agents=[support_agent], tasks=[task])
print(crew.kickoff())

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Use similarity scores from the vector store to determine if retrieved documents are actually relevant
2. Set a threshold — if the best match score is below it, treat it as no match
3. Add a prompt instruction that tells the LLM to say it doesn't have the information rather than guessing


## 7. Evaluation Criteria

Check your solution against these criteria:

- Detects irrelevant retrievals
- Does not hallucinate answers
- Returns clear no-info message
- Still answers relevant queries
